# Type Repair Notebook for Mirrored Cosmos DB Data

Use this notebook after Microsoft Fabric Cosmos DB mirroring when the SQL analytics endpoint shows incompatible values as `NULL`.

The notebook reads the mirrored Delta data directly from OneLake, inspects likely type problems, repairs numeric / JSON / array fields, optionally recovers values from `_raw_body`, and writes a cleansed gold Delta table for downstream BI workloads.

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, DoubleType, StringType, StructField, StructType

SOURCE_DELTA_PATH = "abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>/Tables/cosmos_mirror_sales"
GOLD_TABLE_NAME = "gold.cosmos_sales_repaired"
NULL_ALERT_THRESHOLD = 0.30
WRITE_MODE = "overwrite"  # or "merge" when a business key is available
MERGE_KEY = "id"
RAW_BODY_COLUMN = "_raw_body"

NUMERIC_TYPE_MAP = {
    "orderTotal": "double",
    "taxAmount": "double",
    "itemCount": "int"
}

JSON_TYPE_MAP = {
    "shippingAddress": StructType([
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("postalCode", StringType(), True)
    ])
}

ARRAY_COLUMNS = ["tags", "discountCodes"]
RAW_BODY_RECOVERY_MAP = {
    "orderTotal": "double",
    "taxAmount": "double"
}

In [ ]:
raw_df = spark.read.format("delta").load(SOURCE_DELTA_PATH)
display(raw_df.limit(20))
raw_df.printSchema()

In [ ]:
total_rows = raw_df.count()
null_rate_exprs = [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in raw_df.columns
]
null_counts = raw_df.agg(*null_rate_exprs).collect()[0].asDict()
null_profile = spark.createDataFrame(
    [(c, int(n), round(float(n) / total_rows, 4)) for c, n in null_counts.items()],
    ["column_name", "null_count", "null_pct"]
)
potential_type_issues = null_profile.filter(F.col("null_pct") >= F.lit(NULL_ALERT_THRESHOLD)).orderBy(F.col("null_pct").desc())
display(potential_type_issues)

In [ ]:
repaired_df = raw_df

# 1) Safe numeric coercion for string-or-number payloads.
for column_name, target_type in NUMERIC_TYPE_MAP.items():
    if column_name in repaired_df.columns:
        repaired_df = repaired_df.withColumn(
            column_name,
            F.expr(f"try_cast(`{column_name}` as {target_type})")
        )

# 2) Parse JSON string columns into structs and flatten selected fields.
for column_name, json_schema in JSON_TYPE_MAP.items():
    if column_name in repaired_df.columns:
        parsed_name = f"{column_name}_parsed"
        repaired_df = repaired_df.withColumn(parsed_name, F.from_json(F.col(column_name), json_schema))
        for nested_field in json_schema.fieldNames():
            repaired_df = repaired_df.withColumn(f"{column_name}_{nested_field}", F.col(f"{parsed_name}.{nested_field}"))

# 3) Normalize mixed array types into array<string>.
for column_name in ARRAY_COLUMNS:
    if column_name in repaired_df.columns:
        repaired_df = repaired_df.withColumn(
            column_name,
            F.when(
                F.col(column_name).isNull(),
                F.lit(None).cast(ArrayType(StringType()))
            ).otherwise(
                F.expr(
                    f"filter(transform(`{column_name}`, x -> CASE WHEN x IS NULL THEN NULL WHEN typeof(x) = 'string' THEN cast(x as string) ELSE to_json(x) END), x -> x IS NOT NULL)"
                )
            )
        )

# 4) Recover high-fidelity values from _raw_body when available.
if RAW_BODY_COLUMN in repaired_df.columns:
    for column_name, target_type in RAW_BODY_RECOVERY_MAP.items():
        repaired_df = repaired_df.withColumn(
            column_name,
            F.coalesce(
                F.col(column_name),
                F.expr(f"try_cast(get_json_object(`{RAW_BODY_COLUMN}`, '$.{column_name}') as {target_type})")
            )
        )

display(repaired_df.limit(20))
repaired_df.printSchema()

In [ ]:
if WRITE_MODE == "merge":
    if not spark.catalog.tableExists(GOLD_TABLE_NAME):
        (
            repaired_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(GOLD_TABLE_NAME)
        )
    else:
        target = DeltaTable.forName(spark, GOLD_TABLE_NAME)
        (
            target.alias("t")
            .merge(repaired_df.alias("s"), f"t.{MERGE_KEY} = s.{MERGE_KEY}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
else:
    (
        repaired_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(GOLD_TABLE_NAME)
    )

spark.sql(f"SELECT COUNT(*) AS gold_rows FROM {GOLD_TABLE_NAME}").show()

In [ ]:
before_nulls = raw_df.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in raw_df.columns])
after_nulls = repaired_df.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in repaired_df.columns if c in raw_df.columns])

print(f"Raw row count: {raw_df.count()}")
print(f"Repaired row count: {repaired_df.count()}")

display(before_nulls)
display(after_nulls)
display(
    potential_type_issues.join(
        spark.createDataFrame([(k, v) for k, v in RAW_BODY_RECOVERY_MAP.items()], ["column_name", "recovery_type"]),
        on="column_name",
        how="left"
    )
)